## Verify that gpu is used and download dependencies

In [1]:
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [2]:
!pip install -q transformers accelerate bitsandbytes datasets peft trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.6 MB/s eta 0:00:00


## Set Up the Model

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig, LlamaTokenizerFast

# 1. Define the Model ID from Hugging Face Hub
# Microsoft's Phi-3 is an outstanding ~4B model for structured tasks
model_id = "google/gemma-3-4b-it"
gdrive_cache_dir = "/content/drive/MyDrive/LM_Training_Cache/"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ Hugging Face Token successfully loaded from Colab Secrets.")
except ImportError:
    print("❌ Could not import userdata. Make sure you are running inside Google Colab.")

# 2. Configure 4-bit quantization (QLoRA standard)
# This downloads the weights but instantly maps them to a highly compressed 4-bit structure
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normalized Float 4 (best for LLMs)
    bnb_4bit_compute_dtype=torch.bfloat16, # Keeps execution fast and stable
    bnb_4bit_use_double_quant=True       # Extra compression layer to save VRAM
)

# 3. Download the Tokenizer
print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, revision="main")
# Ensure standard padding configurations for batch training later
tokenizer.pad_token = tokenizer.eos_token

print("Loading model configuration...")
# Load the config first
initial_config = AutoConfig.from_pretrained(model_id, trust_remote_code=True, revision="main")

config = initial_config # Use the (potentially modified) initial_config

# 4. Download and load the Model in 4-bit
print("Downloading and quantizing model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config, # Pass the potentially modified config
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    revision="main",
    cache_dir=gdrive_cache_dir,
    # attn_implementation="eager" # Explicitly use eager attention to bypass rope_scaling issues
)

print("\n🎉 Success! Model downloaded and successfully compressed into 4-bit memory.")
print(f"Memory footprint in VRAM: {model.get_memory_footprint() / 1e6:.2f} MB")

✅ Hugging Face Token successfully loaded from Colab Secrets.
Loading model configuration...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



🎉 Success! Model downloaded and successfully compressed into 4-bit memory.
Memory footprint in VRAM: 3170.62 MB


Now, let's define a path in Google Drive to save the model and tokenizer. We'll check if the model already exists at this path. If not, we'll save it. Otherwise, we'll load it directly from Drive.

## Verify the Baseline

In [5]:
# A dummy sample from your test dataset
test_prompt = """Predict the winner of this League of Legends match based on the team drafts and bans.

Patch: 14.5 | Tier: Diamond II
Blue Team: Top=Aatrox, Jungle=Lee Sin, Mid=Ahri, ADC=Jinx, Support=Thresh
Red Team: Top=Malphite, Jungle=Viego, Mid=Orianna, ADC=Ezreal, Support=Lulu
Blue Bans: Yasuo, Yone, Zed, Blitzcrank, Shaco
Red Bans: Yuumi, K'Sante, Maokai, Karma, Janna

Prediction:"""

# Format into tokens
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

# Generate output
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.1)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Predict the winner of this League of Legends match based on the team drafts and bans.

Patch: 14.5 | Tier: Diamond II
Blue Team: Top=Aatrox, Jungle=Lee Sin, Mid=Ahri, ADC=Jinx, Support=Thresh
Red Team: Top=Malphite, Jungle=Viego, Mid=Orianna, ADC=Ezreal, Support=Lulu
Blue Bans: Yasuo, Yone, Zed, Blitzcrank, Shaco
Red Bans: Yuumi, K'Sante, Maokai, Karma, Janna

Prediction: Blue Team (Aatrox, Lee Sin, Ahri, Jinx


1) Download the db

In [6]:
import gdown
import os

file_id = '1oTwXYj5U7fA0mZkokmiZFSuXy0xN7osU'
url = f'https://drive.google.com/uc?id={file_id}'
DB_PATH = '/content/game_data.db'

if not os.path.exists(DB_PATH):
  gdown.download(url, DB_PATH, quiet=False)
else:
  print('DB already exists')

Downloading...
From: https://drive.google.com/uc?id=1oTwXYj5U7fA0mZkokmiZFSuXy0xN7osU
To: /content/game_data.db
100%|██████████| 401k/401k [00:00<00:00, 52.9MB/s]


2) Retrieve data from db

In [7]:
import sqlite3
import pandas as pd

def retrieve_all():
  conn = sqlite3.connect(DB_PATH)
  df = pd.read_sql_query("SELECT * FROM matches", conn)
  conn.close()
  return df

df = retrieve_all()
df.drop(columns=["cached_at"])
print(df.head())

data_size = len(df)
print(data_size)

     match_id   patch_version  average_tier  blue_win  blue_top  blue_jungle  \
0  5584384055  16.12.788.4269            39         1      13.0         26.0   
1  5584345438  16.12.788.4269            39         0      34.0         15.0   
2  5584331276  16.12.788.4269            39         1      21.0          0.0   
3  5584323002  16.12.788.4269            39         1     110.0        129.0   
4  5584311069  16.12.788.4269            39         0      51.0        120.0   

   blue_mid  blue_adc  blue_support  red_top  ...  blue_ban_3  blue_ban_4  \
0     102.0     115.0          89.0      0.0  ...         145         110   
1      13.0     116.0          17.0    143.0  ...         123          69   
2     151.0     113.0          18.0    142.0  ...         109          15   
3       1.0      24.0          21.0    145.0  ...          -1         115   
4      70.0     151.0          87.0     39.0  ...          52          41   

   red_ban_0  red_ban_1 red_ban_2 red_ban_3 red_ban_4 ga

3. Convert champion indices back to champions' names

In [8]:
import requests

version = df["patch_version"][0][:5] + ".1"
print(version)
url = f"https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json"
data = requests.get(url).json()["data"]

champions = [info["name"] for info in data.values()]
champions = sorted(champions)
print(champions)

16.12.1
['Aatrox', 'Ahri', 'Akali', 'Akshan', 'Alistar', 'Ambessa', 'Amumu', 'Anivia', 'Annie', 'Aphelios', 'Ashe', 'Aurelion Sol', 'Aurora', 'Azir', 'Bard', "Bel'Veth", 'Blitzcrank', 'Brand', 'Braum', 'Briar', 'Caitlyn', 'Camille', 'Cassiopeia', "Cho'Gath", 'Corki', 'Darius', 'Diana', 'Dr. Mundo', 'Draven', 'Ekko', 'Elise', 'Evelynn', 'Ezreal', 'Fiddlesticks', 'Fiora', 'Fizz', 'Galio', 'Gangplank', 'Garen', 'Gnar', 'Gragas', 'Graves', 'Gwen', 'Hecarim', 'Heimerdinger', 'Hwei', 'Illaoi', 'Irelia', 'Ivern', 'Janna', 'Jarvan IV', 'Jax', 'Jayce', 'Jhin', 'Jinx', "K'Sante", "Kai'Sa", 'Kalista', 'Karma', 'Karthus', 'Kassadin', 'Katarina', 'Kayle', 'Kayn', 'Kennen', "Kha'Zix", 'Kindred', 'Kled', "Kog'Maw", 'LeBlanc', 'Lee Sin', 'Leona', 'Lillia', 'Lissandra', 'Lucian', 'Lulu', 'Lux', 'Malphite', 'Malzahar', 'Maokai', 'Master Yi', 'Mel', 'Milio', 'Miss Fortune', 'Mordekaiser', 'Morgana', 'Naafiri', 'Nami', 'Nasus', 'Nautilus', 'Neeko', 'Nidalee', 'Nilah', 'Nocturne', 'Nunu & Willump', 'Olaf',

### Map Champion IDs to Names

As explained, using the `data` dictionary to create a mapping from champion ID (`key`) to champion name is the correct approach. I will now create this mapping and apply it to the relevant columns in your DataFrame.

In [9]:
pick_ban_columns = []
roles = ['top', 'jungle', 'mid', 'adc', 'support']
bans = range(5)

for team in ['blue', 'red']:
    for role in roles:
        pick_ban_columns.append(f"{team}_{role}")
    for ban_num in bans:
        pick_ban_columns.append(f"{team}_ban_{ban_num}")

# Map champion IDs (which are now treated as indices) to names from the 'champions' list
for col in pick_ban_columns:
    # Convert float ID to integer index. Handle NaN values by treating them as -1 or another indicator.
    # Using a lambda function to safely map, handling potential out-of-bounds indices and NaN.
    df[col] = df[col].apply(lambda x:
        champions[int(x)] if pd.notna(x) and 0 <= int(x) < len(champions) else 'Unknown'
    ).astype(str)

display(df.head())

,match_id,patch_version,average_tier,blue_win,blue_top,blue_jungle,blue_mid,blue_adc,blue_support,red_top,...,blue_ban_3,blue_ban_4,red_ban_0,red_ban_1,red_ban_2,red_ban_3,red_ban_4,game_duration,queue_id,cached_at
0,5584384055,16.12.788.4269,39,1,Azir,Diana,Quinn,Senna,Nautilus,Aatrox,...,Vayne,Riven,Lux,Zed,Pyke,Hecarim,Gragas,1239,420,None
1,5584345438,16.12.788.4269,39,0,Fiora,Bel'Veth,Azir,Seraphine,Brand,Urgot,...,Sivir,LeBlanc,Anivia,Graves,Vayne,Xerath,Yasuo,2011,420,None
2,5584331276,16.12.788.4269,39,1,Camille,Aatrox,Viktor,Samira,Braum,Udyr,...,Rengar,Bel'Veth,Poppy,Sivir,LeBlanc,Riven,Kindred,1737,420,None
3,5584323002,16.12.788.4269,39,1,Riven,Sylas,Ahri,Corki,Camille,Vayne,...,Unknown,Senna,Nautilus,Talon,Yasuo,Graves,Renekton,938,420,None
4,5584311069,16.12.788.4269,39,0,Jax,Shyvana,Lee Sin,Viktor,Nami,Gnar,...,Jayce,Graves,Seraphine,Unknown,Vayne,Rengar,Akshan,1139,420,None


### Convert Average Tier Scores to Tier Names

Based on the scoring method provided, I will convert the `average_tier` numerical values into their respective tier and division names. The logic involves mapping the integer quotient of the score by 4 to the tier (e.g., Iron, Bronze) and the remainder to the division (e.g., IV, III, II, I).

In [10]:
def convert_score_to_tier_name(score):
    if pd.isna(score): # Handle potential NaN values
        return 'Unknown'

    # Convert to integer as scores are whole numbers
    score = int(score)

    rank_map = {
        0: "IRON", 1: "BRONZE", 2: "SILVER", 3: "GOLD", 4: "PLATINUM",
        5: "EMERALD", 6: "DIAMOND", 7: "MASTER", 8: "GRANDMASTER", 9: "CHALLENGER"
    }

    division_map = {
        0: "IV", 1: "III", 2: "II", 3: "I"
    }

    tier_index = score // 4
    division_index = score % 4

    tier_name = rank_map.get(tier_index, 'Unknown Tier')
    division_name = division_map.get(division_index, 'Unknown Division')

    return f"{tier_name} {division_name}"

# Apply the conversion function to the 'average_tier' column
df['average_tier_name'] = df['average_tier'].apply(convert_score_to_tier_name)

# Display the DataFrame head with the new column
display(df[['average_tier', 'average_tier_name']].head())

,average_tier,average_tier_name
0,39,CHALLENGER I
1,39,CHALLENGER I
2,39,CHALLENGER I
3,39,CHALLENGER I
4,39,CHALLENGER I


4) Split the dataset into training and testing set

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["blue_win"])
y = df["blue_win"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=40)


### 5) Implement `row_to_prompt` Function

This function will take a single row of the preprocessed DataFrame and construct a natural language prompt that the model can understand. The prompt will describe the match details, including patch, tier, team compositions, and bans, formatted similarly to the example prompt used earlier.

In [12]:
def row_to_prompt(row):
    patch = row['patch_version']
    tier = row['average_tier_name']

    blue_team_picks = f"Top={row['blue_top']}, Jungle={row['blue_jungle']}, Mid={row['blue_mid']}, ADC={row['blue_adc']}, Support={row['blue_support']}"
    red_team_picks = f"Top={row['red_top']}, Jungle={row['red_jungle']}, Mid={row['red_mid']}, ADC={row['red_adc']}, Support={row['red_support']}"

    blue_bans = ", ".join([row[f'blue_ban_{i}'] for i in range(5) if row[f'blue_ban_{i}'] != 'Unknown' and row[f'blue_ban_{i}'] != '-1'])
    red_bans = ", ".join([row[f'red_ban_{i}'] for i in range(5) if row[f'red_ban_{i}'] != 'Unknown' and row[f'red_ban_{i}'] != '-1'])

    prompt = f"""Predict the winner of this League of Legends match based on the team drafts and bans.\n\nPatch: {patch} | Tier: {tier}\nBlue Team: {blue_team_picks}\nRed Team: {red_team_picks}\nBlue Bans: {blue_bans}\nRed Bans: {red_bans}\n\nPrediction:"""
    return prompt

# Example usage:
# Select a row from X_test (e.g., the first row)
sample_row = X_test.iloc[0]
example_prompt = row_to_prompt(sample_row)
print("\n--- Example Prompt ---")
print(example_prompt)


--- Example Prompt ---
Predict the winner of this League of Legends match based on the team drafts and bans.

Patch: 16.12.788.4269 | Tier: GRANDMASTER I
Blue Team: Top=Aatrox, Jungle=Diana, Mid=Nasus, ADC=Caitlyn, Support=Bard
Red Team: Top=Sett, Jungle=Elise, Mid=Yone, ADC=Jhin, Support=Leona
Blue Bans: Thresh, Senna, Rengar, Riven
Red Bans: Bel'Veth, Zoe, Riven, Nautilus, Pyke

Prediction:


### 6) Implement `evaluate_model_on_test_set` Function

This function will iterate through the `X_test` dataset, convert each row to a prompt using the `row_to_prompt` function, and then use the loaded model to generate a prediction. It will compare these predictions against the actual outcomes in `y_test` to calculate and report the accuracy.

In [13]:
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def evaluate_model_on_test_set(model, tokenizer, X_test, y_test, max_new_tokens=15, temperature=0.1):
    correct_predictions = 0
    total_samples = len(X_test)
    predictions = []
    true_labels = []
    predicted_labels = []

    print(f"Evaluating model on {total_samples} test samples...")

    for i in range(total_samples):
        row = X_test.iloc[i]
        true_winner = y_test.iloc[i]

        prompt = row_to_prompt(row)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature)

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract the prediction from the generated text (e.g., looking for 'Blue Team' or 'Red Team')
        predicted_winner = "Unknown"
        if "Blue Team" in generated_text:
            predicted_winner = "Blue Team"
        elif "Red Team" in generated_text:
            predicted_winner = "Red Team"

        # Map true_winner (0 or 1) to "Blue Team" or "Red Team"
        actual_winner_str = "Blue Team" if true_winner == 1 else "Red Team"

        # Store labels for confusion matrix
        true_labels.append(actual_winner_str)
        predicted_labels.append(predicted_winner)

        # Compare prediction with actual winner
        if predicted_winner == actual_winner_str:
            correct_predictions += 1

        predictions.append({
            'sample_id': i,
            'prompt': prompt,
            'generated_text': generated_text,
            'predicted_winner': predicted_winner,
            'actual_winner': actual_winner_str,
            'correct': (predicted_winner == actual_winner_str)
        })

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{total_samples} samples...")

    accuracy = (correct_predictions / total_samples) * 100
    print(f"\nEvaluation complete!")
    print(f"Correct Predictions: {correct_predictions}")
    print(f"Total Samples: {total_samples}")
    print(f"Accuracy: {accuracy:.2f}%")

    return accuracy, predictions, true_labels, predicted_labels

### 7) Model Evaluation and Reporting Utility

To streamline the evaluation process for both the baseline and fine-tuned models, we'll create a single utility function `evaluate_and_save_results`. This function will:

1.  Call `evaluate_model_on_test_set` to get predictions and labels.
2.  Plot a confusion matrix with a dynamic title.
3.  Save the detailed evaluation results to a CSV file in Google Drive with a dynamic filename.

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def evaluate_and_save_results(model_to_eval, tokenizer_to_eval, X_data, y_data, report_name_prefix):
    print(f"--- Evaluating {report_name_prefix.replace('_', ' ').title()} Model ---")
    accuracy, detailed_predictions, true_labels, predicted_labels = evaluate_model_on_test_set(
        model_to_eval, tokenizer_to_eval, X_data, y_data
    )

    # Plot Confusion Matrix
    cm = confusion_matrix(true_labels, predicted_labels, labels=["Blue Team", "Red Team", "Unknown"])
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Blue Team", "Red Team", "Unknown"])
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    ax.set_title(f'Confusion Matrix - {report_name_prefix.replace('_', ' ').title()} Model')
    plt.tight_layout()
    plt.show()

    # Save Detailed Results to CSV
    eval_results_df = pd.DataFrame(detailed_predictions)
    results_save_path = f'/content/drive/MyDrive/model_evaluation_results_{report_name_prefix}.csv'
    eval_results_df.to_csv(results_save_path, index=False)
    print(f"Detailed evaluation results for {report_name_prefix} model saved to {results_save_path}")
    print(f"First 5 rows of {report_name_prefix} results:")
    display(eval_results_df.head())
    print(f"--- Finished Evaluation for {report_name_prefix.replace('_', ' ').title()} Model ---\n")

### 8) Baseline Model Evaluation

Now, let's evaluate the initial (baseline) model's performance using the `evaluate_and_save_results` function.

In [15]:
# Evaluate the baseline model
evaluate_and_save_results(model, tokenizer, X_test, y_test, 'baseline')

--- Evaluating Baseline Model ---
Evaluating model on 985 test samples...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

### 9) Fine-tuning the Model

Now we will fine-tune the `Phi-3-medium-4k-instruct` model using our training data (`X_train`, `y_train`). We will use the `trl` library's `SFTTrainer` for efficient training with LoRA (Low-Rank Adaptation) and 4-bit quantization.

First, we need to prepare our training data into a suitable format for the `SFTTrainer` and define a formatting function that turns each row into a prompt including the expected answer.

In [25]:
from transformers import TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer
from datasets import Dataset

# Combine X_train and y_train for the SFTTrainer dataset
train_df_for_finetuning = X_train.copy()
train_df_for_finetuning['blue_win'] = y_train

# Create a Hugging Face Dataset from the DataFrame
train_dataset = Dataset.from_pandas(train_df_for_finetuning)

# Define the formatting function for SFTTrainer
def formatting_prompts_func(examples):
    row = examples

    patch = row['patch_version']
    tier = row['average_tier_name']

    blue_team_picks = f"Top={row['blue_top']}, Jungle={row['blue_jungle']}, Mid={row['blue_mid']}, ADC={row['blue_adc']}, Support={row['blue_support']}"
    red_team_picks = f"Top={row['red_top']}, Jungle={row['red_jungle']}, Mid={row['red_mid']}, ADC={row['red_adc']}, Support={row['red_support']}"

    # Handle potential 'Unknown' or '-1' for bans
    blue_bans_list = [row[f'blue_ban_{j}'] for j in range(5) if row[f'blue_ban_{j}'] != 'Unknown' and row[f'blue_ban_{j}'] != '-1']
    red_bans_list = [row[f'red_ban_{j}'] for j in range(5) if row[f'red_ban_{j}'] != 'Unknown' and row[f'red_ban_{j}'] != '-1']

    blue_bans = ", ".join(blue_bans_list) if blue_bans_list else "None"
    red_bans = ", ".join(red_bans_list) if red_bans_list else "None"

    actual_winner_str = "Blue Team" if row['blue_win'] == 1 else "Red Team"

    prompt = f"""Predict the winner of this League of Legends match based on the team drafts and bans.\n\nPatch: {patch} | Tier: {tier}\nBlue Team: {blue_team_picks}\nRed Team: {red_team_picks}\nBlue Bans: {blue_bans}\nRed Bans: {red_bans}\n\nPrediction: {actual_winner_str}"""

    return prompt

print("Dataset for fine-tuning created and formatting function defined.")

Dataset for fine-tuning created and formatting function defined.


In [28]:
lora_config = LoraConfig(
    r=16,                       # LoRA attention dimension
    lora_alpha=16,              # Alpha parameter for LoRA scaling
    lora_dropout=0.05,          # Dropout probability for LoRA layers
    bias="none",                # Do not train bias terms
    task_type="CAUSAL_LM",      # Specifies the task
    target_modules='all-linear', # Apply LoRA to all linear layers for simplicity
)

# 2. Training Arguments
training_args = TrainingArguments(
    output_dir="./gemma_finetuned",  # Directory to save checkpoints
    per_device_train_batch_size=4, # Batch size for training
    gradient_accumulation_steps=2, # Number of updates steps to accumulate before performing a backward/update pass
    warmup_steps=10,               # Number of warmup steps for learning rate scheduler
    max_steps=50,                  # Total number of training steps (adjust based on dataset size and desired training time)
    learning_rate=2e-4,            # Learning rate
    fp16=True,                     # Use mixed precision training
    logging_steps=10,              # Log every 10 steps
    optim="paged_adamw_8bit",      # Optimizer
    save_strategy="steps",         # Save checkpoint every `save_steps`
    save_steps=25,                 # Save checkpoint every 25 steps
)

# 3. Initialize SFTTrainer
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    formatting_func=formatting_prompts_func,
    peft_config=lora_config,
    args=training_args,
)

# 4. Start Training
print("Starting fine-tuning...")
sft_trainer.train()
print("Fine-tuning complete!")

# Get the fine-tuned model and tokenizer from the trainer
finetuned_model = sft_trainer.model
finetuned_tokenizer = tokenizer # Tokenizer generally remains the same, but good practice to reference

Applying formatting function to train dataset:   0%|          | 0/2952 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/2952 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2952 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2952 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2952 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Starting fine-tuning...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 14.56 GiB of which 155.81 MiB is free. Including non-PyTorch memory, this process has 14.41 GiB memory in use. Of the allocated memory 14.10 GiB is allocated by PyTorch, and 177.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Save the fine-tuned model and tokenizer to Google Drive
finetuned_model_save_path = '/content/drive/MyDrive/Gemma_3_4b_it_finetuned'

print(f"Saving fine-tuned model and tokenizer to {finetuned_model_save_path}...")
finetuned_model.save_pretrained(finetuned_model_save_path)
finetuned_tokenizer.save_pretrained(finetuned_model_save_path)
print("Fine-tuned model and tokenizer saved successfully to Google Drive!")

### 10) Fine-tuned Model Evaluation

Now, let's evaluate the performance of the fine-tuned model on the test set using our utility function to see the impact of fine-tuning.

In [ ]:
# Evaluate the fine-tuned model
evaluate_and_save_results(finetuned_model, finetuned_tokenizer, X_test, y_test, 'finetuned')